# OLS + Fixed Effects Panel Regression

`married`가 `lwage`에 미치는 ATE를 식별하기 위해, controls와 panel fixed effects를 포함한 OLS를 사용한다.

- **Reference**: [Wooldridge, *Introductory Econometrics* (Ch.13–14)](https://www.cengage.com/c/introductory-econometrics-a-modern-approach-7e-wooldridge/9781337558860/)
- **Data**: `linearmodels.datasets.wage_panel` (Vella & Verbeek, 1998)

In [4]:
import statsmodels.formula.api as smf
from linearmodels.datasets import wage_panel
from linearmodels.panel import PanelOLS

In [5]:
data = wage_panel.load()
panel = data.set_index(["nr", "year"])
data.head()

,nr,year,black,exper,hisp,hours,married,educ,union,lwage,expersq,occupation
0,13,1980,0,1,0,2672,0,14,0,1.197540,1,9
1,13,1981,0,2,0,2320,0,14,1,1.853060,4,9
2,13,1982,0,3,0,2940,0,14,0,1.344462,9,9
3,13,1983,0,4,0,2960,0,14,0,1.433213,16,9
4,13,1984,0,5,0,3071,0,14,0,1.568125,25,5


## OLS with Controls

In [6]:
ols_model = smf.ols('lwage ~ expersq + union + married + hours', data=data).fit()
ols_model.summary().tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,1.5327,0.032,47.842,0.000,1.470,1.595
expersq,0.0012,0.000,6.208,0.000,0.001,0.002
union,0.1679,0.018,9.239,0.000,0.132,0.204
married,0.1966,0.016,11.948,0.000,0.164,0.229
hours,-3.33e-05,1.42e-05,-2.352,0.019,-6.11e-05,-5.54e-06


## Fixed Effects

### Entity Effects

In [7]:
mod = PanelOLS.from_formula(
    "lwage ~ expersq + union + married + hours + EntityEffects", data=panel
)
mod.fit(cov_type='clustered', cluster_entity=True).summary.tables[1]

,Parameter,Std. Err.,T-stat,P-value,Lower CI,Upper CI
expersq,0.0040,0.0002,16.552,0.0000,0.0035,0.0044
union,0.0784,0.0236,3.3225,0.0009,0.0322,0.1247
married,0.1147,0.0220,5.2213,0.0000,0.0716,0.1577
hours,-8.46e-05,2.22e-05,-3.8105,0.0001,-0.0001,-4.107e-05


### Time Effects

In [8]:
mod = PanelOLS.from_formula(
    "lwage ~ expersq + union + married + hours + TimeEffects", data=panel
)
mod.fit(cov_type='clustered', cluster_time=True).summary.tables[1]

,Parameter,Std. Err.,T-stat,P-value,Lower CI,Upper CI
expersq,-0.0021,0.0002,-10.845,0.0000,-0.0025,-0.0017
union,0.1721,0.0210,8.2153,0.0000,0.1311,0.2132
married,0.1634,0.0070,23.221,0.0000,0.1496,0.1772
hours,-6.535e-05,3.629e-05,-1.8010,0.0718,-0.0001,5.789e-06


### Entity + Time Effects

In [9]:
mod = PanelOLS.from_formula(
    "lwage ~ expersq + union + married + hours + EntityEffects + TimeEffects", data=panel
)
mod.fit(cov_type='clustered', cluster_entity=True, cluster_time=True).summary.tables[1]

,Parameter,Std. Err.,T-stat,P-value,Lower CI,Upper CI
expersq,-0.0062,0.0008,-8.1479,0.0000,-0.0077,-0.0047
union,0.0727,0.0228,3.1858,0.0015,0.0279,0.1174
married,0.0476,0.0177,2.6906,0.0072,0.0129,0.0823
hours,-0.0001,3.546e-05,-3.8258,0.0001,-0.0002,-6.614e-05
